In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 17.1 - Overview, paths, and fixed analysis definition
# Purpose:
# Test all 504,889 unique chromosomal unitig presence/absence patterns against
# continuous log2 ceftazidime MIC across the 176 blaTEM-1-only pathogens while
# accounting for genome-wide chromosomal relatedness through K.
#
# The 504,889 patterns were defined from chromosome sequence before this
# association analysis.
#
# A single null mixed model is first fitted:
#
#   y = beta_0 + u + e
#
# where:
#   y = continuous log2 ceftazidime MIC
#   u has covariance sigma_g^2 * K
#   e has covariance sigma_e^2 * I
#
# The null-model covariance is then held fixed while every unique pattern is
# tested by generalized least squares:
#
#   y = beta_0 + beta_1*x + u + e
#
# where x is the presence/absence state of one unique unitig pattern.
#
# Benjamini-Hochberg correction is applied across all 504,889 tests.
# The four patterns examined in Notebooks 15 and 16 are then located in the
# complete result set.
#
# This notebook tests association. It does not make a causal claim.

from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
from scipy import linalg, optimize, stats
from IPython.display import display
PROJECT_ROOT = _repo_root()

RESULTS_TABLE_DIR = PROJECT_ROOT / "05_Results" / "Tables"

NB11_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "11_Unitig_Patterns"
)

NB16_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "16_Formal_Association_Selected_Sequence_Changes"
)

NB17_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "17_Full_Chromosomal_Unitig_Pattern_Association"
)

NB17_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PATTERN_WORDS_FILE = (
    NB11_DIR
    / "11_unique_pattern_words.npz"
)

PATTERN_SAMPLE_ORDER = (
    NB11_DIR
    / "11_pattern_sample_order.csv"
)

PATTERN_METADATA = (
    NB11_DIR
    / "11_unique_pattern_metadata.csv.gz"
)

NB11_QC = (
    RESULTS_TABLE_DIR
    / "11_unitig_pattern_final_QC.csv"
)

NB16_ANALYSIS_DATASET = (
    NB16_DIR
    / "16_analysis_dataset_176.csv.gz"
)

NB16_RESULTS = (
    RESULTS_TABLE_DIR
    / "16_selected_sequence_change_association_results.csv"
)

NB16_QC = (
    RESULTS_TABLE_DIR
    / "16_selected_sequence_change_association_final_QC.csv"
)

NB15_SUMMARY = (
    RESULTS_TABLE_DIR
    / "15_four_pattern_local_sequence_summary.csv"
)

K_FILE = (
    MYDRIVE
    / "Genome_MIC_AMR_Emergence"
    / "04_Population_Structure"
    / "Notebook04"
    / "04_genome_wide_relatedness_matrix.npz"
)

K_INDEX_FILE = (
    MYDRIVE
    / "Genome_MIC_AMR_Emergence"
    / "02_Data_Preparation"
    / "Notebook03"
    / "03_ceftazidime_pathogen_index.csv"
)

NULL_MODEL_RESULTS = (
    RESULTS_TABLE_DIR
    / "17_null_mixed_model_variance_components.csv"
)

ALL_PATTERN_RESULTS = (
    NB17_DIR
    / "17_all_unique_pattern_association_results.csv.gz"
)

SELECTED_PATTERN_RESULTS = (
    RESULTS_TABLE_DIR
    / "17_selected_four_patterns_full_association_results.csv"
)

ASSOCIATION_SUMMARY = (
    RESULTS_TABLE_DIR
    / "17_full_chromosomal_pattern_association_summary.csv"
)

FINAL_QC = (
    RESULTS_TABLE_DIR
    / "17_full_chromosomal_pattern_association_final_QC.csv"
)

COMPLETION_FILE = (
    NB17_DIR
    / "17_FULL_CHROMOSOMAL_PATTERN_ASSOCIATION_COMPLETE.json"
)

EXPECTED_PATHOGENS = 176
EXPECTED_PATTERNS = 504_889
SELECTED_PATTERN_IDS = [6776, 6788, 9107, 11647]

for path in [
    PROJECT_ROOT,
    RESULTS_TABLE_DIR,
    PATTERN_WORDS_FILE,
    PATTERN_SAMPLE_ORDER,
    PATTERN_METADATA,
    NB11_QC,
    NB16_ANALYSIS_DATASET,
    NB16_RESULTS,
    NB16_QC,
    NB15_SUMMARY,
    K_FILE,
    K_INDEX_FILE,
]:
    assert path.exists(), f"Required input not found: {path}"

print("Notebook 17 - Full Chromosomal Unitig Pattern Association with Continuous Ceftazidime MIC")
print("Pathogens expected:", EXPECTED_PATHOGENS)
print("Unique sequence-derived patterns expected:", f"{EXPECTED_PATTERNS:,}")
print("Selected patterns to locate in the complete result set:", SELECTED_PATTERN_IDS)
print("Outcome: continuous log2 ceftazidime MIC")
print("Population structure: genome-wide chromosomal relatedness matrix K")
print("Multiple-testing correction: Benjamini-Hochberg across all unique patterns")
print("No causal claim will be made.")
print("\nTransition: Cell 17.2 will verify and align the phenotype, pattern representation, metadata, and K.")


In [ ]:
#@title Cell 17.2 - Verify and align the phenotype, patterns, metadata, and K
# Purpose:
# Confirm that all 176 pathogens and all 504,889 unique patterns are present,
# and align K to the exact pathogen order used by the pattern representation.

def normalized_column_name(name):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(name).lower(),
    )

def find_biosample_column(df):
    accepted = {
        "biosample",
        "biosampleid",
        "ncbibiosample",
    }

    matches = [
        column
        for column in df.columns
        if normalized_column_name(column) in accepted
    ]

    if len(matches) != 1:
        raise ValueError(
            "Could not identify exactly one BioSample column.\n"
            f"Available columns: {list(df.columns)}\n"
            f"Matches: {matches}"
        )

    return matches[0]

# Verify previous notebooks.
nb11_qc = pd.read_csv(NB11_QC)
nb16_qc = pd.read_csv(NB16_QC)

assert len(nb11_qc) == 1
assert len(nb16_qc) == 1
assert bool(nb11_qc.loc[0, "final_QC_pass"])
assert bool(nb16_qc.loc[0, "final_QC_pass"])

# Load the exact Notebook 11 pathogen order used when the 176-bit patterns
# were packed into three uint64 words.
sample_order = pd.read_csv(
    PATTERN_SAMPLE_ORDER
)

sample_biosample_column = find_biosample_column(
    sample_order
)

pattern_biosamples = (
    sample_order[
        sample_biosample_column
    ]
    .astype(str)
    .tolist()
)

assert len(pattern_biosamples) == EXPECTED_PATHOGENS
assert len(set(pattern_biosamples)) == EXPECTED_PATHOGENS

# Use the verified continuous phenotype dataset from Notebook 16.
analysis_dataset = pd.read_csv(
    NB16_ANALYSIS_DATASET
)

assert "biosample" in analysis_dataset.columns
assert "log2_ceftazidime_MIC" in analysis_dataset.columns

analysis_dataset["biosample"] = (
    analysis_dataset["biosample"]
    .astype(str)
)

assert len(analysis_dataset) == EXPECTED_PATHOGENS
assert analysis_dataset["biosample"].is_unique

phenotype_lookup = (
    analysis_dataset.set_index("biosample")[
        "log2_ceftazidime_MIC"
    ]
    .to_dict()
)

missing_phenotype = [
    biosample
    for biosample in pattern_biosamples
    if biosample not in phenotype_lookup
]

assert not missing_phenotype, (
    "These pattern-order pathogens are missing from the Notebook 16 phenotype:\n"
    + "\n".join(missing_phenotype[:20])
)

y = np.array(
    [
        phenotype_lookup[biosample]
        for biosample in pattern_biosamples
    ],
    dtype=float,
)

assert y.shape == (EXPECTED_PATHOGENS,)
assert np.isfinite(y).all()

# Load the complete sequence-derived pattern representation and metadata.
unique_pattern_words = np.load(
    PATTERN_WORDS_FILE
)["unique_words"]

pattern_metadata = pd.read_csv(
    PATTERN_METADATA
)

assert unique_pattern_words.shape == (
    EXPECTED_PATTERNS,
    3,
)

assert len(pattern_metadata) == EXPECTED_PATTERNS

assert np.array_equal(
    pattern_metadata[
        "pattern_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        EXPECTED_PATTERNS,
        dtype=np.int64,
    ),
), "Pattern metadata is not in exact pattern_id order."

assert pattern_metadata[
    "present_count"
].between(
    1,
    EXPECTED_PATHOGENS - 1,
).all()

# Verify the four selected pattern identifiers are the same as Notebook 15.
nb15_summary = pd.read_csv(
    NB15_SUMMARY
)

observed_selected_ids = sorted(
    nb15_summary[
        "pattern_id"
    ]
    .astype(int)
    .tolist()
)

assert observed_selected_ids == SELECTED_PATTERN_IDS

# Load K and align it to the pattern order.
k_npz = np.load(
    K_FILE
)

square_arrays = []

for key in k_npz.files:
    array = np.asarray(
        k_npz[key]
    )

    if (
        array.ndim == 2
        and array.shape[0] == array.shape[1]
    ):
        square_arrays.append(
            (
                key,
                array.astype(float),
            )
        )

if len(square_arrays) == 1:
    k_array_name, K_full = square_arrays[0]

elif (
    len(square_arrays) > 1
    and "K" in k_npz.files
):
    k_array_name = "K"
    K_full = np.asarray(
        k_npz["K"],
        dtype=float,
    )

else:
    raise ValueError(
        "Could not identify exactly one square K matrix. "
        f"Available arrays: "
        + str(
            {
                key: np.asarray(k_npz[key]).shape
                for key in k_npz.files
            }
        )
    )

k_index = pd.read_csv(
    K_INDEX_FILE
)

k_biosample_column = find_biosample_column(
    k_index
)

k_biosamples = (
    k_index[
        k_biosample_column
    ]
    .astype(str)
    .tolist()
)

assert len(k_biosamples) == K_full.shape[0]
assert len(set(k_biosamples)) == len(k_biosamples)

k_position = {
    biosample: index
    for index, biosample in enumerate(k_biosamples)
}

missing_from_k = [
    biosample
    for biosample in pattern_biosamples
    if biosample not in k_position
]

assert not missing_from_k, (
    "These pathogens are missing from K:\n"
    + "\n".join(missing_from_k[:20])
)

positions = [
    k_position[biosample]
    for biosample in pattern_biosamples
]

K = K_full[
    np.ix_(
        positions,
        positions,
    )
]

assert K.shape == (
    EXPECTED_PATHOGENS,
    EXPECTED_PATHOGENS,
)

assert np.isfinite(K).all()

symmetry_error = float(
    np.max(
        np.abs(
            K - K.T
        )
    )
)

assert symmetry_error < 1e-8

K = (
    K + K.T
) / 2.0

minimum_eigenvalue = float(
    np.linalg.eigvalsh(
        K
    ).min()
)

assert minimum_eigenvalue > -1e-6

print("Notebook 11 QC: PASS")
print("Notebook 16 QC: PASS")
print("Pathogens aligned:", len(pattern_biosamples))
print("Unique patterns:", f"{len(pattern_metadata):,}")
print("Packed pattern array shape:", unique_pattern_words.shape)
print("K array:", k_array_name)
print("Aligned K shape:", K.shape)
print("Minimum K eigenvalue:", minimum_eigenvalue)
print(
    "Continuous log2 ceftazidime MIC range:",
    float(y.min()),
    "to",
    float(y.max()),
)

print("\nSelected pattern metadata:")

display(
    pattern_metadata.loc[
        pattern_metadata[
            "pattern_id"
        ].isin(
            SELECTED_PATTERN_IDS
        )
    ][
        [
            "pattern_id",
            "n_unitigs",
            "present_count",
            "high_MIC_count",
            "remaining_count",
            "category",
        ]
    ]
)

print("\nCell 17.2 complete.")
print("Transition: Cell 17.3 will estimate the null mixed-model covariance from continuous log2 ceftazidime MIC and K.")


In [ ]:
#@title Cell 17.3 - Estimate the null mixed-model covariance
# Purpose:
# Estimate sigma_g^2 and sigma_e^2 by REML from the intercept-only mixed
# model. The resulting covariance is then used for every one of the 504,889
# pattern tests so that all patterns are tested under the same model.

def fit_null_reml(y, K):
    y = np.asarray(
        y,
        dtype=float,
    ).reshape(-1)

    K = np.asarray(
        K,
        dtype=float,
    )

    n = len(y)

    intercept = np.ones(
        (
            n,
            1,
        ),
        dtype=float,
    )

    identity = np.eye(
        n,
        dtype=float,
    )

    def evaluate_ratio(ratio):
        if ratio < 0:
            return None

        covariance_without_scale = (
            identity
            + ratio * K
        )

        try:
            cholesky = linalg.cho_factor(
                covariance_without_scale,
                lower=True,
                check_finite=False,
            )
        except linalg.LinAlgError:
            return None

        inverse_y = linalg.cho_solve(
            cholesky,
            y,
            check_finite=False,
        )

        inverse_intercept = linalg.cho_solve(
            cholesky,
            intercept,
            check_finite=False,
        )

        information = float(
            intercept.T
            @ inverse_intercept
        )

        if information <= 0:
            return None

        beta_0 = float(
            (
                intercept.T
                @ inverse_y
            )
            / information
        )

        residual = (
            y
            - beta_0
        )

        inverse_residual = linalg.cho_solve(
            cholesky,
            residual,
            check_finite=False,
        )

        residual_quadratic = float(
            residual
            @ inverse_residual
        )

        degrees_of_freedom = (
            n - 1
        )

        if residual_quadratic <= 0:
            return None

        sigma_e2 = (
            residual_quadratic
            / degrees_of_freedom
        )

        sigma_g2 = (
            ratio
            * sigma_e2
        )

        log_determinant_covariance_without_scale = (
            2.0
            * np.log(
                np.diag(
                    cholesky[0]
                )
            ).sum()
        )

        objective = 0.5 * (
            degrees_of_freedom
            * np.log(
                sigma_e2
            )
            + log_determinant_covariance_without_scale
            + np.log(
                information
            )
        )

        return {
            "objective": float(
                objective
            ),
            "sigma_g2_to_sigma_e2_ratio": float(
                ratio
            ),
            "sigma_g2": float(
                sigma_g2
            ),
            "sigma_e2": float(
                sigma_e2
            ),
            "intercept": float(
                beta_0
            ),
        }

    def objective_on_log_ratio(
        log_ratio,
    ):
        result = evaluate_ratio(
            np.exp(
                log_ratio
            )
        )

        if result is None:
            return np.inf

        return result[
            "objective"
        ]

    optimized = optimize.minimize_scalar(
        objective_on_log_ratio,
        bounds=(
            -12.0,
            12.0,
        ),
        method="bounded",
        options={
            "xatol": 1e-8,
            "maxiter": 500,
        },
    )

    candidates = []

    zero_result = evaluate_ratio(
        0.0
    )

    if zero_result is not None:
        candidates.append(
            zero_result
        )

    if optimized.success:
        optimized_result = evaluate_ratio(
            float(
                np.exp(
                    optimized.x
                )
            )
        )

        if optimized_result is not None:
            candidates.append(
                optimized_result
            )

    high_result = evaluate_ratio(
        float(
            np.exp(
                12.0
            )
        )
    )

    if high_result is not None:
        candidates.append(
            high_result
        )

    assert candidates, (
        "Null REML optimization failed."
    )

    return min(
        candidates,
        key=lambda item: item[
            "objective"
        ],
    )

null_fit = fit_null_reml(
    y,
    K,
)

sigma_g2 = float(
    null_fit[
        "sigma_g2"
    ]
)

sigma_e2 = float(
    null_fit[
        "sigma_e2"
    ]
)

V = (
    sigma_g2 * K
    + sigma_e2
    * np.eye(
        EXPECTED_PATHOGENS,
        dtype=float,
    )
)

V_cholesky = linalg.cho_factor(
    V,
    lower=True,
    check_finite=False,
)

V_inverse = linalg.cho_solve(
    V_cholesky,
    np.eye(
        EXPECTED_PATHOGENS,
        dtype=float,
    ),
    check_finite=False,
)

V_inverse = (
    V_inverse
    + V_inverse.T
) / 2.0

ones = np.ones(
    EXPECTED_PATHOGENS,
    dtype=float,
)

V_inverse_ones = (
    V_inverse
    @ ones
)

V_inverse_y = (
    V_inverse
    @ y
)

intercept_information = float(
    ones
    @ V_inverse_ones
)

intercept_outcome_product = float(
    ones
    @ V_inverse_y
)

assert intercept_information > 0
assert np.isfinite(V_inverse).all()

variance_fraction = float(
    sigma_g2
    / (
        sigma_g2
        + sigma_e2
    )
)

null_result_table = pd.DataFrame(
    [
        {
            "pathogens": EXPECTED_PATHOGENS,
            "sigma_g2": sigma_g2,
            "sigma_e2": sigma_e2,
            "sigma_g2_to_sigma_e2_ratio": float(
                null_fit[
                    "sigma_g2_to_sigma_e2_ratio"
                ]
            ),
            "chromosomal_relatedness_variance_fraction": variance_fraction,
            "null_model_intercept": float(
                null_fit[
                    "intercept"
                ]
            ),
        }
    ]
)

null_result_table.to_csv(
    NULL_MODEL_RESULTS,
    index=False,
)

display(
    null_result_table
)

print("Null mixed-model covariance: PASS")
print("\nCell 17.3 complete.")
print("Transition: Cell 17.4 will test all 504,889 unique unitig patterns.")


In [ ]:
#@title Cell 17.4 - Test all 504,889 unique unitig patterns
# Purpose:
# Test every unique sequence-derived presence/absence pattern against
# continuous log2 ceftazidime MIC by GLS using the fixed null-model
# covariance estimated in Cell 17.3.
#
# Patterns are processed in blocks so the analysis does not require a dense
# 176 x 504,889 matrix to be kept in memory.

number_of_patterns = len(
    pattern_metadata
)

assert number_of_patterns == EXPECTED_PATTERNS

estimated_log2_MIC_difference = np.empty(
    number_of_patterns,
    dtype=np.float64,
)

standard_error = np.empty(
    number_of_patterns,
    dtype=np.float64,
)

t_statistic = np.empty(
    number_of_patterns,
    dtype=np.float64,
)

p_value_two_sided = np.empty(
    number_of_patterns,
    dtype=np.float64,
)

BLOCK_SIZE = 10_000
degrees_of_freedom = (
    EXPECTED_PATHOGENS
    - 2
)

analysis_start_time = time.time()

for block_start in range(
    0,
    number_of_patterns,
    BLOCK_SIZE,
):
    block_end = min(
        block_start
        + BLOCK_SIZE,
        number_of_patterns,
    )

    block_words = unique_pattern_words[
        block_start:block_end
    ]

    block_pattern_count = (
        block_end
        - block_start
    )

    # Decode the 176 presence/absence states from the three uint64 words.
    pattern_states = np.empty(
        (
            block_pattern_count,
            EXPECTED_PATHOGENS,
        ),
        dtype=np.float64,
    )

    for pathogen_index in range(
        EXPECTED_PATHOGENS
    ):
        word_index = (
            pathogen_index
            // 64
        )

        bit_index = (
            pathogen_index
            % 64
        )

        pattern_states[
            :,
            pathogen_index,
        ] = (
            (
                block_words[
                    :,
                    word_index,
                ]
                >> np.uint64(
                    bit_index
                )
            )
            & np.uint64(
                1
            )
        )

    observed_presence_counts = (
        pattern_states.sum(
            axis=1
        )
    )

    expected_presence_counts = (
        pattern_metadata.loc[
            block_start:block_end - 1,
            "present_count",
        ]
        .to_numpy(
            dtype=float
        )
    )

    assert np.array_equal(
        observed_presence_counts,
        expected_presence_counts,
    ), (
        f"Presence-count mismatch in patterns "
        f"{block_start} to {block_end - 1}."
    )

    pattern_intercept_product = (
        pattern_states
        @ V_inverse_ones
    )

    pattern_outcome_product = (
        pattern_states
        @ V_inverse_y
    )

    pattern_states_times_inverse = (
        pattern_states
        @ V_inverse
    )

    pattern_information = np.einsum(
        "ij,ij->i",
        pattern_states_times_inverse,
        pattern_states,
    )

    determinant = (
        intercept_information
        * pattern_information
        - pattern_intercept_product
        * pattern_intercept_product
    )

    if np.any(
        determinant <= 0
    ):
        bad_indices = np.flatnonzero(
            determinant <= 0
        )

        raise ValueError(
            "Non-positive GLS information determinant for pattern IDs: "
            + ", ".join(
                str(
                    block_start
                    + int(index)
                )
                for index in bad_indices[:20]
            )
        )

    beta_1 = (
        (
            intercept_information
            * pattern_outcome_product
        )
        - (
            pattern_intercept_product
            * intercept_outcome_product
        )
    ) / determinant

    beta_1_variance = (
        intercept_information
        / determinant
    )

    beta_1_standard_error = np.sqrt(
        beta_1_variance
    )

    beta_1_t = (
        beta_1
        / beta_1_standard_error
    )

    beta_1_p = (
        2.0
        * stats.t.sf(
            np.abs(
                beta_1_t
            ),
            df=degrees_of_freedom,
        )
    )

    estimated_log2_MIC_difference[
        block_start:block_end
    ] = beta_1

    standard_error[
        block_start:block_end
    ] = beta_1_standard_error

    t_statistic[
        block_start:block_end
    ] = beta_1_t

    p_value_two_sided[
        block_start:block_end
    ] = beta_1_p

    del (
        pattern_states,
        pattern_states_times_inverse,
    )

    if (
        block_start == 0
        or block_end == number_of_patterns
        or block_end % 50_000 == 0
    ):
        elapsed_minutes = (
            time.time()
            - analysis_start_time
        ) / 60.0

        print(
            f"Tested {block_end:,} / {number_of_patterns:,} patterns "
            f"({elapsed_minutes:.1f} min elapsed)"
        )

assert np.isfinite(
    estimated_log2_MIC_difference
).all()

assert np.isfinite(
    standard_error
).all()

assert np.isfinite(
    t_statistic
).all()

assert np.isfinite(
    p_value_two_sided
).all()

assert (
    (
        p_value_two_sided
        >= 0
    )
    & (
        p_value_two_sided
        <= 1
    )
).all()

critical_t = stats.t.ppf(
    0.975,
    df=degrees_of_freedom,
)

CI95_lower = (
    estimated_log2_MIC_difference
    - critical_t
    * standard_error
)

CI95_upper = (
    estimated_log2_MIC_difference
    + critical_t
    * standard_error
)

print(
    "\nAll unique patterns tested:",
    f"{number_of_patterns:,}",
)

print(
    "Elapsed time:",
    f"{(time.time() - analysis_start_time) / 60.0:.1f} minutes",
)

print("\nCell 17.4 complete.")
print("Transition: Cell 17.5 will correct across all 504,889 tests and locate the four selected patterns.")


In [ ]:
#@title Cell 17.5 - Correct across all tests and locate the four selected patterns
# Purpose:
# Apply Benjamini-Hochberg correction across all 504,889 tests, rank the
# complete results, and extract the four selected patterns.

def benjamini_hochberg(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    ranked_p_values = (
        p_values[
            order
        ]
    )

    adjusted_ranked = (
        ranked_p_values
        * number_of_tests
        / np.arange(
            1,
            number_of_tests + 1,
            dtype=float,
        )
    )

    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[
            ::-1
        ]
    )[
        ::-1
    ]

    adjusted_ranked = np.clip(
        adjusted_ranked,
        0.0,
        1.0,
    )

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    adjusted[
        order
    ] = adjusted_ranked

    return adjusted

Benjamini_Hochberg_q_value = (
    benjamini_hochberg(
        p_value_two_sided
    )
)

p_value_rank = np.empty(
    number_of_patterns,
    dtype=np.int64,
)

p_value_rank[
    np.argsort(
        p_value_two_sided
    )
] = np.arange(
    1,
    number_of_patterns + 1,
    dtype=np.int64,
)

all_results = pattern_metadata[
    [
        "pattern_id",
        "first_unitig_index",
        "n_unitigs",
        "present_count",
        "high_MIC_count",
        "remaining_count",
        "category",
    ]
].copy()

all_results[
    "estimated_log2_MIC_difference"
] = estimated_log2_MIC_difference

all_results[
    "standard_error"
] = standard_error

all_results[
    "CI95_lower"
] = CI95_lower

all_results[
    "CI95_upper"
] = CI95_upper

all_results[
    "t_statistic"
] = t_statistic

all_results[
    "degrees_of_freedom"
] = degrees_of_freedom

all_results[
    "p_value_two_sided"
] = p_value_two_sided

all_results[
    "Benjamini_Hochberg_q_value"
] = Benjamini_Hochberg_q_value

all_results[
    "p_value_rank_among_all_patterns"
] = p_value_rank

all_results[
    "association_after_Benjamini_Hochberg_0_05"
] = (
    all_results[
        "Benjamini_Hochberg_q_value"
    ]
    <= 0.05
)

all_results[
    "estimated_direction"
] = np.where(
    all_results[
        "estimated_log2_MIC_difference"
    ]
    > 0,
    "higher_log2_MIC",
    np.where(
        all_results[
            "estimated_log2_MIC_difference"
        ]
        < 0,
        "lower_log2_MIC",
        "no_difference",
    ),
)

all_results.to_csv(
    ALL_PATTERN_RESULTS,
    index=False,
    compression="gzip",
)

selected_unitig_lookup = (
    pd.read_csv(
        NB15_SUMMARY
    )[
        [
            "pattern_id",
            "unitig_id",
            "gene_or_nearest_gene_descriptions",
        ]
    ]
    .copy()
)

selected_results = (
    all_results.loc[
        all_results[
            "pattern_id"
        ].isin(
            SELECTED_PATTERN_IDS
        )
    ]
    .merge(
        selected_unitig_lookup,
        on="pattern_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "pattern_id"
    )
    .reset_index(
        drop=True
    )
)

assert len(
    selected_results
) == len(
    SELECTED_PATTERN_IDS
)

selected_results[
    "selected_pattern_survives_full_Benjamini_Hochberg_correction"
] = (
    selected_results[
        "Benjamini_Hochberg_q_value"
    ]
    <= 0.05
)

selected_results.to_csv(
    SELECTED_PATTERN_RESULTS,
    index=False,
)

number_significant_all = int(
    (
        all_results[
            "Benjamini_Hochberg_q_value"
        ]
        <= 0.05
    ).sum()
)

number_selected_surviving = int(
    selected_results[
        "selected_pattern_survives_full_Benjamini_Hochberg_correction"
    ].sum()
)

number_selected_surviving_with_higher_MIC = int(
    (
        selected_results[
            "selected_pattern_survives_full_Benjamini_Hochberg_correction"
        ]
        & (
            selected_results[
                "estimated_log2_MIC_difference"
            ]
            > 0
        )
    ).sum()
)

print(
    "Unique patterns tested:",
    f"{number_of_patterns:,}",
)

print(
    "Patterns with Benjamini-Hochberg q <= 0.05:",
    f"{number_significant_all:,}",
)

print(
    "Selected patterns surviving correction across all patterns:",
    number_selected_surviving,
    "/",
    len(
        SELECTED_PATTERN_IDS
    ),
)

print(
    "Selected patterns surviving correction with a positive estimated MIC difference:",
    number_selected_surviving_with_higher_MIC,
)

print("\nResults for the four selected patterns:")

display(
    selected_results[
        [
            "pattern_id",
            "unitig_id",
            "present_count",
            "estimated_log2_MIC_difference",
            "CI95_lower",
            "CI95_upper",
            "p_value_two_sided",
            "Benjamini_Hochberg_q_value",
            "p_value_rank_among_all_patterns",
            "estimated_direction",
            "selected_pattern_survives_full_Benjamini_Hochberg_correction",
            "gene_or_nearest_gene_descriptions",
        ]
    ]
)

print("\nTop 10 patterns from the complete sequence-derived analysis:")

display(
    all_results.sort_values(
        [
            "p_value_two_sided",
            "pattern_id",
        ]
    )[
        [
            "pattern_id",
            "n_unitigs",
            "present_count",
            "estimated_log2_MIC_difference",
            "CI95_lower",
            "CI95_upper",
            "p_value_two_sided",
            "Benjamini_Hochberg_q_value",
            "p_value_rank_among_all_patterns",
        ]
    ].head(
        10
    )
)

print("\nCell 17.5 complete.")
print("Transition: Cell 17.6 will independently verify the selected pattern calculations and state the final interpretation.")


In [ ]:
#@title Cell 17.6 - Independent verification and final interpretation
# Purpose:
# Recalculate the four selected patterns directly from their packed
# presence/absence states and verify that they match the complete analysis.
# Then state the stopping decision.

def decode_one_pattern(
    pattern_words,
):
    states = np.zeros(
        EXPECTED_PATHOGENS,
        dtype=float,
    )

    for pathogen_index in range(
        EXPECTED_PATHOGENS
    ):
        word_index = (
            pathogen_index
            // 64
        )

        bit_index = (
            pathogen_index
            % 64
        )

        states[
            pathogen_index
        ] = (
            (
                pattern_words[
                    word_index
                ]
                >> np.uint64(
                    bit_index
                )
            )
            & np.uint64(
                1
            )
        )

    return states

verification_rows = []

for pattern_id in SELECTED_PATTERN_IDS:
    x = decode_one_pattern(
        unique_pattern_words[
            pattern_id
        ]
    )

    X = np.column_stack(
        [
            np.ones(
                EXPECTED_PATHOGENS,
                dtype=float,
            ),
            x,
        ]
    )

    information = (
        X.T
        @ V_inverse
        @ X
    )

    outcome_product = (
        X.T
        @ V_inverse
        @ y
    )

    beta = np.linalg.solve(
        information,
        outcome_product,
    )

    covariance_beta = np.linalg.inv(
        information
    )

    selected_beta = float(
        beta[
            1
        ]
    )

    selected_standard_error = float(
        np.sqrt(
            covariance_beta[
                1,
                1,
            ]
        )
    )

    selected_t = (
        selected_beta
        / selected_standard_error
    )

    selected_p = float(
        2.0
        * stats.t.sf(
            abs(
                selected_t
            ),
            df=degrees_of_freedom,
        )
    )

    stored_row = (
        selected_results.loc[
            selected_results[
                "pattern_id"
            ]
            == pattern_id
        ]
        .iloc[
            0
        ]
    )

    assert np.isclose(
        selected_beta,
        float(
            stored_row[
                "estimated_log2_MIC_difference"
            ]
        ),
        rtol=1e-10,
        atol=1e-10,
    )

    assert np.isclose(
        selected_standard_error,
        float(
            stored_row[
                "standard_error"
            ]
        ),
        rtol=1e-10,
        atol=1e-10,
    )

    assert np.isclose(
        selected_p,
        float(
            stored_row[
                "p_value_two_sided"
            ]
        ),
        rtol=1e-10,
        atol=1e-12,
    )

    verification_rows.append(
        {
            "pattern_id": pattern_id,
            "decoded_present_count": int(
                x.sum()
            ),
            "direct_GLS_estimated_log2_MIC_difference": selected_beta,
            "direct_GLS_standard_error": selected_standard_error,
            "direct_GLS_p_value_two_sided": selected_p,
            "matches_complete_analysis": True,
        }
    )

verification = pd.DataFrame(
    verification_rows
)

display(
    verification
)

all_four_verified = bool(
    verification[
        "matches_complete_analysis"
    ].all()
)

assert all_four_verified

number_selected_surviving = int(
    selected_results[
        "selected_pattern_survives_full_Benjamini_Hochberg_correction"
    ].sum()
)

number_selected_positive_surviving = int(
    (
        selected_results[
            "selected_pattern_survives_full_Benjamini_Hochberg_correction"
        ]
        & (
            selected_results[
                "estimated_log2_MIC_difference"
            ]
            > 0
        )
    ).sum()
)

summary_row = {
    "pathogens_tested": EXPECTED_PATHOGENS,
    "unique_sequence_patterns_tested": number_of_patterns,
    "patterns_with_Benjamini_Hochberg_q_le_0_05": number_significant_all,
    "selected_patterns_examined": len(
        SELECTED_PATTERN_IDS
    ),
    "selected_patterns_surviving_full_Benjamini_Hochberg_correction": number_selected_surviving,
    "selected_patterns_surviving_with_higher_log2_MIC": number_selected_positive_surviving,
    "selected_pattern_calculations_independently_verified": all_four_verified,
    "causal_claim_made": False,
}

pd.DataFrame(
    [
        summary_row
    ]
).to_csv(
    ASSOCIATION_SUMMARY,
    index=False,
)

final_qc_row = {
    "Notebook_11_QC_pass": True,
    "Notebook_16_QC_pass": True,
    "pathogen_count_correct": True,
    "pattern_count_correct": True,
    "all_patterns_tested": True,
    "Benjamini_Hochberg_correction_across_all_patterns_completed": True,
    "selected_pattern_direct_GLS_verification_pass": all_four_verified,
    "final_QC_pass": True,
}

pd.DataFrame(
    [
        final_qc_row
    ]
).to_csv(
    FINAL_QC,
    index=False,
)

COMPLETION_FILE.write_text(
    json.dumps(
        {
            "status": "complete",
            "pathogens_tested": EXPECTED_PATHOGENS,
            "unique_sequence_patterns_tested": int(
                number_of_patterns
            ),
            "patterns_with_Benjamini_Hochberg_q_le_0_05": int(
                number_significant_all
            ),
            "selected_patterns_surviving_full_Benjamini_Hochberg_correction": int(
                number_selected_surviving
            ),
            "selected_patterns_surviving_with_higher_log2_MIC": int(
                number_selected_positive_surviving
            ),
            "final_QC_pass": True,
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("\nFinal selected-pattern results:")

display(
    selected_results[
        [
            "pattern_id",
            "unitig_id",
            "present_count",
            "estimated_log2_MIC_difference",
            "CI95_lower",
            "CI95_upper",
            "p_value_two_sided",
            "Benjamini_Hochberg_q_value",
            "p_value_rank_among_all_patterns",
            "selected_pattern_survives_full_Benjamini_Hochberg_correction",
        ]
    ]
)

print("\nFinal QC: PASS")

if number_selected_surviving == 0:
    print(
        "\nStopping decision: none of the four selected sequence changes "
        "survived Benjamini-Hochberg correction across all 504,889 unique "
        "sequence-derived patterns. Stop here for these four changes."
    )

else:
    print(
        "\nStopping decision:",
        number_selected_surviving,
        "of the four selected sequence changes survived Benjamini-Hochberg "
        "correction across all 504,889 unique sequence-derived patterns."
    )

    print(
        "A surviving change with a positive estimated effect can be described "
        "as associated with higher ceftazidime MIC after accounting for "
        "genome-wide chromosomal relatedness through K and correction across "
        "the complete set of unique sequence-derived patterns."
    )

print(
    "\nThis remains an observational association in the same 176 pathogens. "
    "It is not a causal result."
)
